# Geo model QC (standard template)

Run by `opmagent` for every geo model version built from tops and logs; the HTML export is the version's QC report and the approval is tied to its sha256. **Read only:** the notebook loads the build payload (tops, the logs used, the upscaled values, the spec), the tables from the database and every cell array from its registered file, computes checks and plots, and writes `qc_summary.json`. Don't edit this template for one model: write an investigation notebook instead (`opmagent geo investigate`).

In [ ]:
payload_path = "payload.json"   # written by the geo pipeline
db_path = ""                    # SQLite database (opened read-only)
version_id = 0                  # geo_model_versions.id
summary_path = "qc_summary.json"

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'   # self-contained HTML export
import pandas as pd
from IPython.display import display
from opmagent.notebooks import geo_qc_lib as qc

data = qc.load(payload_path, db_path, version_id)
findings = qc.run_all_checks(data)
qc.header(data)

## Summary

Every QC finding, worst first. Fail and warn findings don't block the build; they are for the reviewer.

In [ ]:
qc.show_findings(findings)

## Assumptions and validator warnings

In [ ]:
display(qc.show_list(data.get('assumptions', []), title='Assumptions'))
qc.show_list(title='Validator warnings', items=[w['message'] + (f" ({w['where']})" if w.get('where') else '') for w in data.get('validator_warnings', [])])

## 1. Surfaces and well tops

Each surface as gridded (cell centres) with the wells' tops. The misfit is the gridded depth at the well's column minus the top; it grows with dip and with the well's distance from the cell centre. Zones pinched to the minimum thickness are where the surfaces were stopped from crossing.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'surfaces']))
if data['layer_depths'] is not None:
    display(qc.fig_surfaces(data))
    display(qc.tops_misfit(data).round(2))

## 2. Logs vs model at the wells

The build upscales each well's logs per layer (porosity arithmetic, permeability geometric, NTG from the Vsh cut-off) and kriges them between the wells, so the well's cell should hold the upscaled values; it differs by what the map changes between the well and the cell centre (`offset_m`). *Leave one well out* predicts each well from the others: it shows how well the wells constrain the map between them.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'wells']))
df = qc.at_wells(data)
if len(df):
    display(qc.fig_at_wells(data))
    display(df[['well_name', 'layer', 'offset_m', 'phi_log', 'phi_model', 'perm_log', 'perm_model', 'ntg_log', 'ntg_model']].round(4))

In [ ]:
for w in list(qc.well_cells(data)['well_name'])[:12]:   # the first 12 wells
    if (data['logs']['well_name'].astype(str) == w).any():
        display(qc.fig_well(data, w))

## 3. Cell properties

Maps of the top, middle and bottom layers and vertical sections through the first well, with the zone surfaces (black), layer boundaries (grey) and the spec's contacts.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'properties']))
for kw, log in (('PORO', False), ('PERMX', True), ('NTG', False)):
    if kw in data['arrays']:
        display(qc.fig_layer_maps(data, kw, log))
if data['layer_depths'] is not None:
    display(qc.fig_section(data, 'PORO', along='x'))
    display(qc.fig_section(data, 'PORO', along='y'))
    display(qc.fig_section(data, 'PERMX', along='x', log=True))

In [ ]:
display(qc.fig_distributions(data))
if 'PERMX' in data['arrays']:
    display(qc.fig_poroperm(data))
stats = pd.DataFrame([{'keyword': k, 'min': a.min(), 'mean': a.mean(), 'max': a.max()} for k, a in data['arrays'].items() if hasattr(a, 'ndim') and a.ndim == 3])
display(stats.round(4))

## 4. Volumes

Pore volume per zone recomputed from the stored arrays (not the build's own numbers). The HCPV uses the contacts given in the spec, for orientation only: the contacts that initialise the model are the res model's.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'volumes']))
if data['layer_depths'] is not None and 'PORO' in data['arrays']:
    v = qc.volumes(data)
    display(v.style.format({'bulk_rm3': '{:,.0f}', 'pore_volume_rm3': '{:,.0f}', 'hcpv_rm3': '{:,.0f}', 'mean_phi': '{:.3f}', 'mean_ntg': '{:.2f}'}))

## 5. Files

The grid and every cell array the deck builder will include, checked against the sha256 they were registered with.

In [ ]:
display(qc.show_findings([f for f in findings if f['area'] == 'files']))
pd.DataFrame(data['files'])

## Result

In [ ]:
summary = qc.write_summary(summary_path, findings, data)
print(f"Overall: {summary['overall'].upper()}  " + ", ".join(f'{k} {v}' for k, v in summary['counts'].items()))